In [8]:
import pandas as pd

url = "https://raw.githubusercontent.com/Alam-A99/DATAAKSI/refs/heads/main/dataset_fiskal_97_hipoFull.csv"

# Membaca dataset
df = pd.read_csv(url)

# Memfilter semua baris yang mengandung teks "Surabaya" (tidak peka huruf besar/kecil)
Surabaya_data = df[df.apply(lambda row: row.astype(str).str.contains('Surabaya', case=False).any(), axis=1)]

# Menampilkan hasil
print("Kota Surabaya:")
display(Surabaya_data)

Kota Surabaya:


,Kota,Tahun,DBH_PAJAK,DBH_SDA,DBH_Lainnya,DAU_Grant,DAU_Earmark,DAK_Fisik,DAK_Nonfisik,PAD,...,bm_lingkungan,bm_ekonomi,bm_infrastruktur,bm_pariwisata,bm_perlindungan_sosial,KEMISKINAN,PENGANGGURAN,KETIMPANGAN,IPM,PE
605,Kota Surabaya,2014,124.81,61.39,102.76,239.44,196.29,197.37,187.73,247.59,...,157.81,109.37,152.78,50.06,258.32,17.33,8.38,0.42,68.43,3.19
606,Kota Surabaya,2015,123.07,110.77,86.95,258.82,161.85,149.58,180.02,417.14,...,136.88,124.97,108.58,110.31,112.49,7.46,5.67,0.33,75.62,7.77
607,Kota Surabaya,2016,230.70,93.01,116.44,246.38,191.42,126.24,226.14,365.36,...,62.59,210.70,54.91,102.06,202.38,15.04,8.82,0.33,60.96,5.07
608,Kota Surabaya,2017,169.66,81.15,94.73,335.24,133.02,164.49,133.26,366.51,...,63.19,200.24,269.85,112.62,196.30,18.43,9.88,0.43,64.04,6.40
609,Kota Surabaya,2018,208.35,89.83,132.03,378.91,237.96,234.86,164.94,332.82,...,223.00,224.02,291.40,251.48,142.43,15.32,4.54,0.39,79.55,4.47
610,Kota Surabaya,2019,205.51,182.73,130.00,375.42,173.72,180.16,185.94,315.86,...,164.48,140.63,135.39,119.92,171.51,9.10,4.47,0.39,82.71,4.13
611,Kota Surabaya,2020,210.92,145.57,67.17,358.33,192.31,167.20,215.36,308.32,...,106.63,256.00,134.63,97.88,128.45,15.40,3.28,0.47,71.20,4.44
612,Kota Surabaya,2021,182.04,175.46,132.09,368.67,241.14,217.09,282.34,446.56,...,203.49,101.02,51.77,176.07,85.46,17.63,8.61,0.38,65.75,4.46
613,Kota Surabaya,2022,262.07,104.42,109.38,402.69,194.23,235.66,178.75,318.37,...,113.21,167.26,220.06,266.41,103.45,16.92,3.69,0.35,81.38,5.27
614,Kota Surabaya,2023,285.68,214.26,91.82,295.52,246.88,210.09,159.38,494.48,...,194.41,250.23,256.36,110.34,76.38,5.47,9.09,0.33,63.18,5.26


In [9]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.stats.api as sms
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy import stats
import plotly.graph_objects as go
import plotly.io as pio

# Memaksa Plotly untuk merender grafik interaktif langsung di dalam sel Google Colab
pio.renderers.default = 'colab'

# 1. MEMBACA DATA LANGSUNG DARI LINK GITHUB (RAW URL)
url = "https://raw.githubusercontent.com/Alam-A99/DATAAKSI/main/dataset_fiskal_97_hipoFull.csv"
df_all = pd.read_csv(url)

# Filter data untuk Kota Surabaya
df = df_all[df_all['Kota'] == 'Kota Surabaya'].copy()

print(f"Total data Surabaya yang dianalisis: {len(df)} baris.\n")

# 2. MODEL REGRESI & UJI ASUMSI KLASIK

X = df[['bm_ekonomi', 'DBH_PAJAK']]
Y = df['KEMISKINAN']
X_sm = sm.add_constant(X)
model = sm.OLS(Y, X_sm).fit()

# Menampilkan Summary OLS
print(model.summary())

print("\n" + "="*50)
print("FUNGSI REGRESI (Data Asli):")
print(f"Y = {model.params['const']:.4f} + {model.params['bm_ekonomi']:.4f}(BM_Ekonomi) - {abs(model.params['DBH_PAJAK']):.4f}(DBH_PAJAK)")
print("="*50)

print("\nHASIL UJI ASUMSI KLASIK")

# 1. Normalitas
shapiro_stat, shapiro_p = stats.shapiro(model.resid)
print(f"- Normalitas (Shapiro-Wilk) P-Value: {shapiro_p:.4f} (>0.05 = Lolos)")

# 2. Multikolinearitas (VIF)
vif_data = pd.DataFrame()
vif_data["Variabel"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print(f"- Multikolinearitas (VIF)            : {vif_data['VIF'].max():.4f} (<10 = Lolos)")

# 3. Heteroskedastisitas
bp_test = sms.het_breuschpagan(model.resid, model.model.exog)
print(f"- Heteroskedastisitas (BP) P-Value   : {bp_test[1]:.4f} (>0.05 = Lolos)")

# 4. Autokorelasi
dw_value = sm.stats.stattools.durbin_watson(model.resid)
print(f"- Autokorelasi (Durbin-Watson)       : {dw_value:.4f} (Mendekati 2 = Lolos)\n")

# 3. VISUALISASI 3D INTERAKTIF

# Membuat grid bidang regresi berdasarkan rentang data asli
x_range = np.linspace(df['bm_ekonomi'].min(), df['bm_ekonomi'].max(), 20)
y_range = np.linspace(df['DBH_PAJAK'].min(), df['DBH_PAJAK'].max(), 20)
x_grid, y_grid = np.meshgrid(x_range, y_range)

# Menghitung prediksi Z untuk permukaan bidang
z_grid = model.params['const'] + (model.params['bm_ekonomi'] * x_grid) + (model.params['DBH_PAJAK'] * y_grid)

# Inisialisasi Plotly Figure
fig = go.Figure()

# Menambahkan Bidang Regresi Permukaan
fig.add_trace(go.Surface(
    x=x_grid,
    y=y_grid,
    z=z_grid,
    name='Bidang Regresi',
    colorscale='viridis',
    opacity=0.8,
    showscale=False
))

# Menambahkan Titik Data Aktual Asli
fig.add_trace(go.Scatter3d(
    x=df['bm_ekonomi'],
    y=df['DBH_PAJAK'],
    z=df['KEMISKINAN'],
    mode='markers',
    name='Data Aktual',
    marker=dict(size=6, color='red', line=dict(color='white', width=1))
))

# Mengatur Tampilan Layout Grafik
fig.update_layout(
    title=f"Plot Bidang Regresi 3D: KEMISKINAN terhadap BM Ekonomi & DBH PAJAK (Surabaya) | R² = {model.rsquared:.2f}",
    template='plotly_dark',
    scene=dict(
        xaxis_title='BM Ekonomi',
        yaxis_title='DBH_PAJAK',
        zaxis_title='KEMISKINAN'
    ),
    margin=dict(l=0, r=0, b=0, t=40)
)

# Menampilkan kanvas interaktif di bawah sel Colab
fig.show()

Total data Surabaya yang dianalisis: 11 baris.

                            OLS Regression Results                            
Dep. Variable:             KEMISKINAN   R-squared:                       0.037
Model:                            OLS   Adj. R-squared:                 -0.204
Method:                 Least Squares   F-statistic:                    0.1534
Date:                Thu, 15 Oct 2026   Prob (F-statistic):              0.860
Time:                        08:28:36   Log-Likelihood:                -31.265
No. Observations:                  11   AIC:                             68.53
Df Residuals:                       8   BIC:                             69.72
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
cons